# R3-04 · Agentes — Profundización (opcional) 🔬

**Formación Pública — Rama R3 · IA Aplicada · Notebook de profundización**

Opcional y de **transferencia**: le das al agente una herramienta nueva sobre otro corpus —
**buscar en las 245 fichas de ChileAtiende** (tu RAG de R3-03, hecho herramienta)— y lo
blindas con **guardas**: argumentos válidos, calidad del dato y privacidad. Además **mides**
al planificador con un set de peticiones, igual que mediste la recuperación con hit@k.

Todo corre sin red ni LLM: el agente usa su planificador de reglas.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import os, re, json, unicodedata, urllib.request
import pandas as pd

# "En vivo o caché": usa los archivos locales; si faltan (ej. Colab), se bajan del repo.
for archivo in ["aire_diario.csv", "estaciones.csv", "matricula.csv", "fichas.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)

aire = pd.read_csv("aire_diario.csv", parse_dates=["fecha"])   # PM2.5 diario SINCA (rama R2)
estaciones = pd.read_csv("estaciones.csv")                      # dónde está cada estación
matricula = pd.read_csv("matricula.csv")                        # matrícula 2025 MINEDUC (rama R1)

def normalizar(texto):
    """minúsculas y sin tildes, para comparar textos escritos por personas."""
    t = unicodedata.normalize("NFKD", str(texto).lower())
    return "".join(c for c in t if not unicodedata.combining(c))
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
fichas = pd.read_csv("fichas.csv")

# ── Herramientas y agente de la lección (versión resuelta) ──
NORMA_PM25_DIARIA = 50   # µg/m³: valor de la norma primaria diaria de PM2.5 en Chile (DS 12/2011 MMA)

def pm25_promedio(estacion, anio):
    """PM2.5 promedio de un año (µg/m³) en una estación SINCA."""
    d = aire[(aire["estacion"] == estacion) & (aire["fecha"].dt.year == anio)]["pm25"].dropna()
    if d.empty:
        raise ValueError(f"no hay datos de PM2.5 para «{estacion}» en {anio}")
    return {"estacion": estacion, "anio": anio, "pm25_promedio": round(float(d.mean()), 1),
            "dias_con_dato": int(d.size)}

def matricula_comuna(comuna):
    """Establecimientos y estudiantes matriculados (2025) en una comuna."""
    d = matricula[matricula["comuna"].map(normalizar) == normalizar(comuna)]
    if d.empty:
        raise ValueError(f"no conozco la comuna «{comuna}»")
    return {"comuna": d["comuna"].iloc[0], "establecimientos": int(len(d)),
            "estudiantes": int(d["mat_total"].sum())}

BANDEJA_SALIDA = []   # simula el correo institucional: aquí «llegan» las alertas enviadas

def enviar_alerta(destinatario, mensaje):
    """ACCIÓN CON EFECTO: envía (simulado) una alerta a un destinatario."""
    BANDEJA_SALIDA.append({"para": destinatario, "mensaje": mensaje})
    return {"enviado_a": destinatario}

def dias_sobre_norma(estacion, anio, umbral=NORMA_PM25_DIARIA):
    """Días de un año con PM2.5 sobre el umbral (por defecto, 50 µg/m³) en una estación."""
    d = aire[(aire["estacion"] == estacion) & (aire["fecha"].dt.year == anio)]["pm25"].dropna()
    if d.empty:
        raise ValueError(f"no hay datos de PM2.5 para «{estacion}» en {anio}")
    return {"estacion": estacion, "anio": anio, "dias_sobre_norma": int((d > umbral).sum()),
            "dias_con_dato": int(d.size)}

# El «menú» del agente: qué herramientas existen, qué argumentos piden y si tienen EFECTO
# (cambian algo en el mundo). El agente no puede usar nada que no esté aquí.
HERRAMIENTAS = {
    "pm25_promedio":    {"funcion": pm25_promedio,    "args": ["estacion", "anio"], "efecto": False,
                         "descripcion": "PM2.5 promedio anual (µg/m³) de una estación SINCA, años 2020-2025."},
    "dias_sobre_norma": {"funcion": dias_sobre_norma, "args": ["estacion", "anio"], "efecto": False,
                         "descripcion": "Días del año con PM2.5 sobre 50 µg/m³ en una estación, años 2020-2025."},
    "matricula_comuna": {"funcion": matricula_comuna, "args": ["comuna"], "efecto": False,
                         "descripcion": "Establecimientos y estudiantes matriculados en 2025 en una comuna."},
    "enviar_alerta":    {"funcion": enviar_alerta,    "args": ["destinatario", "mensaje"], "efecto": True,
                         "descripcion": "ENVÍA una alerta. Acción con efecto: requiere aprobación humana."},
}

# Del texto de la petición a los argumentos: estación, año y comuna.
ALIAS_ESTACION = {}
comunas_con_estacion = estaciones["comuna"].map(normalizar).value_counts()
for est, com in zip(estaciones["estacion"], estaciones["comuna"]):
    ALIAS_ESTACION[normalizar(re.sub(r"\s*\(.*\)", "", est))] = est   # «El Bosque (Acreditada)» → «el bosque»
    if comunas_con_estacion[normalizar(com)] == 1:                      # la comuna sirve de alias si no es ambigua
        ALIAS_ESTACION.setdefault(normalizar(com), est)

def extraer_estacion(texto):
    t = normalizar(texto)
    for alias in sorted(ALIAS_ESTACION, key=len, reverse=True):
        if re.search(rf"\b{re.escape(alias)}\b", t):
            return ALIAS_ESTACION[alias]
    return None

def extraer_anio(texto):
    m = re.search(r"\b(20\d\d)\b", texto)
    return int(m.group(1)) if m else None

COMUNAS = sorted(matricula["comuna"].unique(), key=len, reverse=True)
def extraer_comuna(texto):
    t = normalizar(texto)
    for c in COMUNAS:
        if re.search(rf"\b{re.escape(normalizar(c))}\b", t):
            return c
    return None

def planificar_reglas(peticion, traza):
    """Planificador de RESPALDO, determinista: decide el siguiente paso con reglas simples."""
    t = normalizar(peticion)
    hechos = [p for p in traza if p["estado"] == "ok"]
    if any(p["estado"] in ("error", "bloqueado_por_guarda") for p in traza):   # algo falló: no insistir
        return {"herramienta": "fin", "args": {}}
    if hechos:                                               # ya tenemos el dato
        quiere_avisar = any(w in t for w in ["avisa", "alerta", "notifica"])
        ya_propuesta = any(p["herramienta"] == "enviar_alerta" for p in traza)
        if quiere_avisar and not ya_propuesta:
            return {"herramienta": "enviar_alerta",
                    "args": {"destinatario": "SEREMI de Medio Ambiente",
                             "mensaje": f"Resultado de la consulta: {hechos[-1]['resultado']}"}}
        return {"herramienta": "fin", "args": {}}
    # Primer paso: ¿qué dato hay que consultar?
    if "norma" in t or "episodio" in t:
        return {"herramienta": "dias_sobre_norma",
                "args": {"estacion": extraer_estacion(peticion), "anio": extraer_anio(peticion)}}
    if "pm2" in t or "aire" in t or "contaminacion" in t:
        return {"herramienta": "pm25_promedio",
                "args": {"estacion": extraer_estacion(peticion), "anio": extraer_anio(peticion)}}
    if "matricula" in t or "estudiantes" in t or "colegios" in t:
        return {"herramienta": "matricula_comuna", "args": {"comuna": extraer_comuna(peticion)}}
    return {"herramienta": "fin", "args": {}}

def ejecutar_paso(plan, traza):
    """Ejecuta UNA herramienta y deja el registro en la traza (sin caerse si falla)."""
    nombre, args = plan["herramienta"], plan["args"]
    registro = {"paso": len(traza) + 1, "herramienta": nombre, "args": args,
                "origen": plan.get("origen", "manual")}
    try:
        registro["resultado"] = HERRAMIENTAS[nombre]["funcion"](**args)
        registro["estado"] = "ok"
    except Exception as e:
        registro["resultado"] = f"{type(e).__name__}: {e}"
        registro["estado"] = "error"
    traza.append(registro)
    return registro

def ejecutar_supervisado(plan, traza, aprobador=None):
    """Si la herramienta tiene EFECTO, solo se ejecuta con aprobación humana explícita."""
    if HERRAMIENTAS[plan["herramienta"]]["efecto"]:
        if aprobador is None or not aprobador(plan):         # por defecto: NO
            registro = {"paso": len(traza) + 1, "herramienta": plan["herramienta"],
                        "args": plan["args"], "origen": plan.get("origen", "manual"),
                        "resultado": "acción NO ejecutada: sin aprobación humana",
                        "estado": "rechazado"}
            traza.append(registro)
            return registro
        registro = ejecutar_paso(plan, traza)
        registro["aprobado_por_humano"] = True
        return registro
    return ejecutar_paso(plan, traza)

def agente(peticion, planificador=None, aprobador=None, max_pasos=4):
    planificador = planificador or planificar_reglas      # aquí, sin LLM
    traza, estado = [], "limite_de_pasos"
    for _ in range(max_pasos):
        plan = planificador(peticion, traza)
        if plan["herramienta"] == "fin":
            estado = "terminado"
            break
        ejecutar_supervisado(plan, traza, aprobador)
    return {"estado": estado, "traza": traza}

# ── Índice de fichas (receta de la profundización de R3-03: título x3 + n-gramas de caracteres) ──
def trocear(texto, max_palabras=120):
    trozos, actual, n = [], [], 0
    for linea in texto.split("\n"):
        if not linea.strip():                    # las líneas en blanco no aportan
            continue
        palabras = len(linea.split())
        if actual and (linea.startswith("## ") or n + palabras > max_palabras):
            trozos.append("\n".join(actual))
            actual, n = [], 0
        actual.append(linea)
        n += palabras
    if actual:
        trozos.append("\n".join(actual))
    return trozos

trozos = pd.DataFrame([{"id": f.id, "titulo": f.titulo, "url": f.url, "texto": t}
                       for f in fichas.itertuples() for t in trocear(f.texto)])
vect_fichas = TfidfVectorizer(strip_accents="unicode", analyzer="char_wb", ngram_range=(3, 5))
X_fichas = vect_fichas.fit_transform((trozos["titulo"] + "\n") * 3 + trozos["texto"])
print("Agente listo con", len(HERRAMIENTAS), "herramientas | índice de fichas:", X_fichas.shape)

## 1. Una herramienta nueva: buscar en las fichas

Escribe `buscar_ficha(consulta)`: transforma la consulta con `vect_fichas`, calcula la similitud con `X_fichas` y devuelve la ficha del **mejor trozo**: `id`, `titulo`, `url` y `puntaje` (redondeado a 2 decimales). Después regístrala en `HERRAMIENTAS` — sin efecto: solo consulta.

### ✍️ Ejercicio 1 — Escribe y registra `buscar_ficha`

In [ ]:
def buscar_ficha(consulta):
    # TODO: puntajes = similitud de la consulta con cada trozo (X_fichas @ vect_fichas.transform([consulta]).T)
    puntajes = ...
    # TODO: i = posición del mayor puntaje (int(np.argmax(puntajes)))
    i = ...
    f = trozos.iloc[i]
    return {"id": int(f["id"]), "titulo": f["titulo"], "url": f["url"], "puntaje": round(float(puntajes[i]), 2)}

# TODO: registra la herramienta en HERRAMIENTAS["buscar_ficha"] con funcion, args=["consulta"],
#       efecto=False y una descripcion
...

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    r = buscar_ficha("¿Cómo me afilio a Fonasa?")
    assert r["id"] == 9715 and r["url"].endswith("/9715"), "Fonasa debería llevar a la ficha 9715"
    assert buscar_ficha("¿Cómo saco la licencia para pescar como deporte?")["id"] == 4636
    assert "buscar_ficha" in HERRAMIENTAS and HERRAMIENTAS["buscar_ficha"]["efecto"] is False
    assert HERRAMIENTAS["buscar_ficha"]["args"] == ["consulta"]
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'puntajes = (X_fichas @ vect_fichas.transform([consulta]).T).toarray().ravel(); i = int(np.argmax(puntajes)); y HERRAMIENTAS["buscar_ficha"] = {...}.')
    print("   Detalle:", e)

## 2. Enseñarle al planificador… y medirlo

Escribe `planificar_v2`: si es el **primer paso** y la petición suena a **trámite** (alguna de `PALABRAS_TRAMITE`), propone `buscar_ficha` con la petición como consulta; en cualquier otro caso, delega en `planificar_reglas`. Luego **mídelo** con un set de peticiones y su herramienta esperada: la fracción de aciertos es tu «hit@1» del planificador.

### ✍️ Ejercicio 2 — Escribe `planificar_v2` y calcula su exactitud

In [ ]:
PALABRAS_TRAMITE = ["como saco", "como obtengo", "como pido", "tramite", "requisitos",
                    "postular", "beca", "bono", "certificado", "beneficio"]

def planificar_v2(peticion, traza):
    t = normalizar(peticion)
    # TODO: si la traza está vacía y alguna palabra de PALABRAS_TRAMITE está en t →
    #       return {"herramienta": "buscar_ficha", "args": {"consulta": peticion}}
    ...
    return planificar_reglas(peticion, traza)

PRUEBAS_PLAN = [
    ("¿Cómo saco el certificado de antecedentes?", "buscar_ficha"),
    ("¿Qué requisitos tiene el bono de invierno?", "buscar_ficha"),
    ("¿Cómo postular a la beca indígena?", "buscar_ficha"),
    ("¿Cuántos días sobre la norma tuvo Nielol en 2021?", "dias_sobre_norma"),
    ("¿Cómo estuvo el aire en Coyhaique en 2022?", "pm25_promedio"),
    ("PM2.5 promedio de Quintero en 2020", "pm25_promedio"),
    ("¿Cuántos estudiantes hay en Valdivia?", "matricula_comuna"),
    ("¿Cuántos colegios tiene Putre?", "matricula_comuna"),
    ("¿Qué beca hay para estudiantes con discapacidad?", "buscar_ficha"),
    ("¿Quién ganó el partido de ayer?", "fin"),
]
# TODO: aciertos = lista de True/False (¿planificar_v2(p, [])["herramienta"] == esperada?)
aciertos = ...
# TODO: exactitud = fracción de aciertos
exactitud = ...
print("exactitud del planificador:", exactitud)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert planificar_v2("¿Cómo saco el certificado de antecedentes?", [])["herramienta"] == "buscar_ficha"
    assert planificar_v2("¿Cuántos estudiantes hay en Valdivia?", [])["herramienta"] == "matricula_comuna"
    assert planificar_v2("¿Cómo saco el certificado de antecedentes?", [{"estado": "ok", "herramienta": "buscar_ficha"}])["herramienta"] == "fin"
    assert abs(exactitud - sum(planificar_v2(p, [])["herramienta"] == e for p, e in PRUEBAS_PLAN) / len(PRUEBAS_PLAN)) < 1e-9
    assert exactitud >= 0.8
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'if not traza and any(w in t for w in PALABRAS_TRAMITE): return {"herramienta": "buscar_ficha", "args": {"consulta": peticion}}.')
    print("   Detalle:", e)

## 3. Guarda de argumentos: no ejecutar a ciegas

El planificador puede proponer una estación `None` («Temuco» es ambiguo), un año sin datos (2030) o una comuna inexistente. Mejor detectarlo **antes** de ejecutar y explicarlo. Escribe `validar_args(plan)` que devuelva `(True, "")` o `(False, motivo)`. Luego `ejecutar_con_guardas` lo usa: si no pasa, registra el paso como `bloqueado_por_guarda`.

### ✍️ Ejercicio 3 — Escribe `validar_args`

In [ ]:
ANIOS_CON_DATOS = set(aire["fecha"].dt.year.unique())     # 2020–2025

def validar_args(plan):
    a = plan["args"]
    # TODO 1: si "estacion" está en a y no es una estación de estaciones["estacion"] →
    #         return False, "estación no reconocida o ambigua ..."
    # TODO 2: si "anio" está en a y no está en ANIOS_CON_DATOS →
    #         return False, f"no hay datos para el año {a['anio']} ..."
    # TODO 3: si "comuna" está en a y es None → return False, "no reconocí la comuna"
    ...
    return True, ""

def ejecutar_con_guardas(plan, traza, aprobador=None):
    ok, motivo = validar_args(plan)
    if not ok:
        traza.append({"paso": len(traza) + 1, "herramienta": plan["herramienta"], "args": plan["args"],
                      "origen": plan.get("origen", "manual"), "resultado": motivo, "estado": "bloqueado_por_guarda"})
        return traza[-1]
    return ejecutar_supervisado(plan, traza, aprobador)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    ok, motivo = validar_args({"herramienta": "pm25_promedio", "args": {"estacion": "Nielol", "anio": 2030}})
    assert ok is False and "2030" in motivo, "un año sin datos debe bloquearse, nombrando el año"
    ok, motivo = validar_args({"herramienta": "pm25_promedio", "args": {"estacion": None, "anio": 2024}})
    assert ok is False, "una estación None (ambigua) debe bloquearse"
    assert validar_args({"herramienta": "matricula_comuna", "args": {"comuna": None}})[0] is False
    assert validar_args({"herramienta": "pm25_promedio", "args": {"estacion": "Nielol", "anio": 2022}}) == (True, "")
    t = []
    ejecutar_con_guardas({"herramienta": "pm25_promedio", "args": {"estacion": "Nielol", "anio": 2030}}, t)
    assert t[-1]["estado"] == "bloqueado_por_guarda"
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'if "anio" in a and a["anio"] not in ANIOS_CON_DATOS: return False, f"no hay datos para el año {a[\'anio\']}"; igual para estación y comuna.')
    print("   Detalle:", e)

## 4. Guarda de calidad: un cero que no es cero

Pregúntale al agente por **El Bosque en 2025**: responde **0 días** sobre la norma. ¿Buena noticia? No: esa estación solo tiene **59 días con dato** ese año (en 2024 tuvo 45 días sobre la norma). Un agente honesto **advierte** cuando el dato es incompleto. Escribe `con_advertencia(resultado, minimo_dias=300)`: si el resultado trae `dias_con_dato` bajo el mínimo, devuelve una copia con la clave `advertencia`.

### ✍️ Ejercicio 4 — Escribe `con_advertencia`

In [ ]:
def con_advertencia(resultado, minimo_dias=300):
    # TODO: si resultado es dict y resultado.get("dias_con_dato", minimo_dias) < minimo_dias,
    #       devuelve {**resultado, "advertencia": f"solo {resultado['dias_con_dato']} días con dato: ..."}
    ...
    return resultado

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    r25 = con_advertencia(dias_sobre_norma("El Bosque (Acreditada)", 2025))
    assert "advertencia" in r25 and "59" in r25["advertencia"], "El Bosque 2025 tiene solo 59 días con dato: debe advertirlo"
    r24 = con_advertencia(dias_sobre_norma("El Bosque (Acreditada)", 2024))
    assert "advertencia" not in r24, "con un año casi completo no hay advertencia"
    assert con_advertencia({"comuna": "Arica", "estudiantes": 1}) == {"comuna": "Arica", "estudiantes": 1}, "resultados sin días quedan igual"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'if isinstance(resultado, dict) and resultado.get("dias_con_dato", minimo_dias) < minimo_dias: return {**resultado, "advertencia": ...}.')
    print("   Detalle:", e)

## 5. Desafío: guarda de privacidad y el agente blindado

Si la petición trae un **RUT**, no debe salir hacia un LLM externo ni quedar en una traza que muchos pueden leer. Escribe `contiene_rut(texto)` y `agente_blindado(peticion, ...)`: si hay RUT, devuelve `{"estado": "bloqueado_privacidad", "traza": []}` **sin planificar nada**; si no, corre el agente usando `planificar_v2`, `ejecutar_con_guardas` y `con_advertencia` en cada resultado.

### ✍️ Ejercicio 5 — Escribe `contiene_rut` y `agente_blindado`

In [ ]:
def contiene_rut(texto):
    # TODO: busca un RUT con la expresión regular r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b"
    return ...

def agente_blindado(peticion, aprobador=None, max_pasos=4):
    # TODO: si contiene_rut(peticion) → return {"estado": "bloqueado_privacidad", "traza": []}
    ...
    traza, estado = [], "limite_de_pasos"
    for _ in range(max_pasos):
        plan = planificar_v2(peticion, traza)
        if plan["herramienta"] == "fin":
            estado = "terminado"
            break
        # TODO: ejecuta con ejecutar_con_guardas y pasa su resultado por con_advertencia
        ...
    return {"estado": estado, "traza": traza}

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    assert contiene_rut("Mi RUT es 12.345.678-5") and contiene_rut("rut 9876543-k")
    assert not contiene_rut("¿Cuántos estudiantes hay en Arica en 2025?")
    assert agente_blindado("¿Cómo pido la pensión de alimentos? Mi RUT es 12.345.678-5") == {"estado": "bloqueado_privacidad", "traza": []}
    r = agente_blindado("¿Cuántos días sobre la norma tuvo El Bosque en 2025?")
    assert r["estado"] == "terminado" and "advertencia" in r["traza"][0]["resultado"], "el resultado de El Bosque 2025 debe venir con advertencia"
    r = agente_blindado("¿Cuál fue el PM2.5 de Nielol en 2030?")
    assert r["traza"][0]["estado"] == "bloqueado_por_guarda"
    assert agente_blindado("¿Cómo saco el certificado de antecedentes?")["traza"][0]["resultado"]["id"] == 3442
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'return re.search(PATRON, texto) is not None; en el bucle: registro = ejecutar_con_guardas(...); registro["resultado"] = con_advertencia(registro["resultado"]).')
    print("   Detalle:", e)

## Para llevar

- Agregar una **herramienta** = una función + una línea en el menú. Agregar **criterio** = enseñarle
  al planificador cuándo usarla… y **medirlo** con un set de peticiones.
- Las **guardas** van **antes** (argumentos válidos, privacidad) y **después** (calidad del
  resultado) de cada herramienta. Ninguna depende del LLM.
- Un **cero con 59 días de dato** no es un cero: un agente público advierte cuando el dato no da
  para concluir.
- La detección de RUT por expresión regular es una **primera barrera**, no una garantía: los datos
  personales se protegen también con diseño (qué se pide, qué se guarda, quién ve la traza).

---

*Fin de la profundización de R3-04.*